In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import rasterio
from rasterio import plot
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
import math
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.interpolate import griddata

In [6]:
from utils.site import Site
from utils.tree import Tree
from utils.constants import ETHZ_COCOA_MAP_THRESHOLDED_FILEPATH, ETHZ_COCOA_MAP_FILEPATH, SILVER_FOLDERPATH, ETHZ_COCOA_MAP_UINT8_FILEPATH

In [4]:
# Check resolution => 1px == 10m x 10m
with rasterio.open(str(ETHZ_COCOA_MAP_FILEPATH)) as src:
    pixel_width, pixel_height = src.res
    
    print(f"Raw pixel width: {pixel_width}")
    print(f"Raw pixel height: {pixel_height}")
    print(f"CRS: {src.crs}") # EPSG:4326 is geographic

    # Figure out pixel actual size (meter)
    bounds = src.bounds
    center_lat = bounds.top # (bounds.bottom + bounds.top) / 2
        
    # Calculate meters per degree of longitude at this specific latitude
    # Formula: 111320.0 * cos(latitude) https://fr.wikipedia.org/wiki/Latitude
    meters_per_degree_lon = 111320.0 * math.cos(math.radians(center_lat))
    
    width_m = pixel_width * meters_per_degree_lon
    height_m = pixel_height * 111320.0
        
    print(f"Based on the map's center latitude ({center_lat:.2f}°)")
    print(f"One pixel: {width_m:.2f} meters wide.")
    print(f"One pixel: {height_m:.2f} meters wide.")

Raw pixel width: 9.055018635493447e-05
Raw pixel height: 9.055023064754131e-05
CRS: EPSG:4326
Based on the map's center latitude (11.17°)
One pixel: 9.89 meters wide.
One pixel: 10.08 meters wide.


In [5]:
UINT8_NODATA = 255 # uint8 NO-DATA value

with rasterio.open(ETHZ_COCOA_MAP_FILEPATH) as src:
    
    # Allocate empty uint8 array 
    binned_data = np.empty(src.shape, dtype=np.uint8)
    
    # Iterate through the file block-by-block to avoid RAM overload
    n_window = len(list(src.block_windows(bidx=1)))
    for idx, (_, window) in enumerate(src.block_windows(bidx=1)):
        print(f"Processing: {idx+1}/{n_window}", end="\r")
        chunk = src.read(indexes=1, window=window, masked=True)
        
        # Bin the data: multiply by 100 and round to keep 2 digits of precision
        # Example: 0.876 -> 87.6 -> 88.0
        chunk_scaled = np.round(chunk * 100)
        
        # Handle missing data 
        chunk_binned = chunk_scaled.filled(UINT8_NODATA).astype(np.uint8)
        
        binned_data[window.toslices()] = chunk_binned

print("Data loaded successfully as uint8!")
print(f"Memory used by array: {binned_data.nbytes / (1024**3):.2f} GB")

Data loaded successfully as uint8!
Memory used by array: 7.58 GB


In [ ]:
# Dump to disk
with rasterio.open(ETHZ_COCOA_MAP_FILEPATH) as src:
    profile = src.profile

profile.update(dtype=binned_data.dtype, count=1, nodata=UINT8_NODATA, compress='lzw')

with rasterio.open(ETHZ_COCOA_MAP_UINT8_FILEPATH, 'w', **profile) as dst:
    dst.write(binned_data, 1)

print(f"Dumped: {ETHZ_COCOA_MAP_UINT8_FILEPATH}")